# HDB Resale Flat Prices — ETL Pipeline (Jan 2012 – Dec 2017)

This notebook runs the pipeline stage by stage. The logic lives in `src/hdb_pipeline/`; the notebook orchestrates it and explains each step.

| Stage | Output group | Status |
|---|---|---|
| 1. Extract | **Raw** | ✅ |
| 2. Standardise | Staging (`master.parquet`) | ✅ |
| 3. Profile | (report input) | ✅ |
| 4. Validate | **Quarantined** | ✅ |
| 5. Enrich (remaining lease, price anomalies) | **Cleaned** | ✅ |
| 6. Transform (Resale Identifier) | **Transformed** | ✅ |
| 7. De-identify | **Hashed** | ✅ |

## 0. Setup

In [1]:
import logging
import sys
from pathlib import Path

import pandas as pd

# Make the package importable without installing it (if you ran `pip install -e .`, this is harmless)
sys.path.insert(0, str(Path.cwd() / "src"))

from hdb_pipeline.config import load_config
from hdb_pipeline.extract import run_extract

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(name)s | %(message)s")
pd.set_option("display.max_columns", 50, "display.width", 200)

CFG = load_config()
CFG["scope"]

{'start_month': '2012-01', 'end_month': '2017-12'}

## 1. Extract → Raw

**What it does**
1. Reads the collection's metadata from the data.gov.sg API to get its child dataset IDs (nothing hardcoded).
2. Reads each dataset's coverage window and keeps only those overlapping the configured scope (`2012-01` to `2017-12`).
3. Downloads each in-scope file through the `initiate-download` → `poll-download` flow, streaming the bytes to `data/raw/` **as-is**.
4. Records lineage in `data/raw/_manifest.json`: SHA-256, size, row count, header and source timestamps.

**Re-runs** use the cached file unless the source has been updated since the last pull. Set `FORCE=True` to re-download everything. Set `OFFLINE=True` to skip the network and only check the committed raw files against their checksums.

In [2]:
FORCE = False
OFFLINE = False

manifest = run_extract(CFG, force=FORCE, offline=OFFLINE)
manifest[["dataset_id", "name", "coverage_start", "coverage_end", "row_count", "bytes", "sha256"]]

2026-10-18 17:53:25,840 INFO    hdb_pipeline.extract | Collection 189 has 5 datasets
2026-10-18 17:53:26,188 INFO    hdb_pipeline.extract | CACHE d_8b84c4ee58e3cfc0ece0d773c8ca6abc  Resale flat prices based on registration date from Jan-2017 onwards unchanged since last pull
2026-10-18 17:53:26,515 INFO    hdb_pipeline.extract | CACHE d_43f493c6c50d54243cc1eab0df142d6a  Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012 unchanged since last pull
2026-10-18 17:53:26,739 INFO    hdb_pipeline.extract | CACHE d_2d5ff9ea31397b66239f245f57751537  Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014 unchanged since last pull
2026-10-18 17:53:26,958 INFO    hdb_pipeline.extract | SKIP  d_ebc5ab87086db484f88045b47411ebc5  Resale Flat Prices (Based on Approval Date), 1990 - 1999 [1990-01..1999-12] out of scope
2026-10-18 17:53:27,194 INFO    hdb_pipeline.extract | CACHE d_ea9ed51da2787afaf8e51f827c304208  Resale Flat Prices (Based on Registration Date), From Jan 2

,dataset_id,name,coverage_start,coverage_end,row_count,bytes,sha256
0,d_43f493c6c50d54243cc1eab0df142d6a,"Resale Flat Prices (Based on Approval Date), 2...",2000-01,2012-02,369651,29369945,5f6a72bd7b9120281863beb1fc4c89aa5f4923f00b0f6c...
1,d_2d5ff9ea31397b66239f245f57751537,Resale Flat Prices (Based on Registration Date...,2012-03,2014-12,52203,4160771,6a16e2dc78f70048ec1a522b59a9727581c36e45e02022...
2,d_ea9ed51da2787afaf8e51f827c304208,Resale Flat Prices (Based on Registration Date...,2015-01,2016-12,37153,3070924,4ff7ce4a4f642fb384d2b75a42e75b0477e50005f4aaa9...
3,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,Resale flat prices based on registration date ...,2017-01,2026-10,242955,24032414,d401fc2e7d2a083576dd388b5ea3d170c2027d2503542d...


### 1a. First look at the raw files

These are read-only checks to see what landed before any processing. Things to watch for:
- **Schema drift**: do all files share the same columns?
- **Scope**: how many rows in each file fall inside the 2012-01 to 2017-12 window? The 2000 – Feb 2012 file holds the Jan 2012 authoritative set.

In [3]:
# Column presence matrix across raw files
all_cols = sorted({c for cols in manifest["columns"] for c in cols},
                  key=lambda c: min(cols.index(c) for cols in manifest["columns"] if c in cols))
schema = pd.DataFrame({row["name"]: [c in row["columns"] for c in all_cols] for _, row in manifest.iterrows()},
                      index=all_cols)
schema.replace({True: "✓", False: "—"})

,"Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012","Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014","Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016",Resale flat prices based on registration date from Jan-2017 onwards
month,✓,✓,✓,✓
town,✓,✓,✓,✓
flat_type,✓,✓,✓,✓
block,✓,✓,✓,✓
street_name,✓,✓,✓,✓
storey_range,✓,✓,✓,✓
floor_area_sqm,✓,✓,✓,✓
flat_model,✓,✓,✓,✓
lease_commence_date,✓,✓,✓,✓
remaining_lease,—,—,✓,✓


In [4]:
# Rows per file, and rows within scope. Read as strings so nothing is coerced
raw = {r["name"]: pd.read_csv(CFG["paths"]["raw"] / r["file"], dtype=str, keep_default_na=False)
       for _, r in manifest.iterrows()}

start, end = CFG["scope"]["start_month"], CFG["scope"]["end_month"]
pd.DataFrame([{
    "file": name,
    "rows_total": len(df),
    "rows_in_scope": df["month"].between(start, end).sum(),
    "min_month": df["month"].min(),
    "max_month": df["month"].max(),
} for name, df in raw.items()])

,file,rows_total,rows_in_scope,min_month,max_month
0,"Resale Flat Prices (Based on Approval Date), 2...",369651,3188,2000-01,2012-02
1,Resale Flat Prices (Based on Registration Date...,52203,52203,2012-03,2014-12
2,Resale Flat Prices (Based on Registration Date...,37153,37153,2015-01,2016-12
3,Resale flat prices based on registration date ...,242955,20509,2017-01,2026-10


In [5]:
# Sample rows from each file
for name, df in raw.items():
    print(f"\n=== {name} ===")
    display(df.head(3))


=== Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012 ===


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
0,2000-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69,Improved,1986,147000
1,2000-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61,Improved,1986,144000
2,2000-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73,New Generation,1976,159000



=== Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014 ===


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
0,2012-03,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45,Improved,1986,250000
1,2012-03,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44,Improved,1980,265000
2,2012-03,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68,New Generation,1980,315000



=== Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016 ===


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2015-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,07 TO 09,60,Improved,1986,70,255000
1,2015-01,ANG MO KIO,3 ROOM,541,ANG MO KIO AVE 10,01 TO 03,68,New Generation,1981,65,275000
2,2015-01,ANG MO KIO,3 ROOM,163,ANG MO KIO AVE 4,01 TO 03,69,New Generation,1980,64,285000



=== Resale flat prices based on registration date from Jan-2017 onwards ===


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,Improved,1979,61 years 04 months,232000
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,New Generation,1978,60 years 07 months,250000
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,New Generation,1980,62 years 05 months,262000


## 2. Standardise → Staging master dataset

Combines the in-scope raw rows into **one master dataset shaped like the Jan 2012 file** (requirement 1).

| Step | Rule | Why |
|---|---|---|
| Authoritative schema | Taken from the dataset whose coverage contains `scope.start_month` (2012-01), i.e. the *2000 – Feb 2012* file | Jan 2012 is the strict authoritative set. Worked out from the manifest, not hardcoded |
| Read | Every column read as text, with `keep_default_na=False` | No silent type changes on read. Values are seen exactly as published |
| Scope | Keep `2012-01 ≤ month ≤ 2017-12`. Rows whose month can't be parsed are **kept** | An unparseable month can't be scoped. Keeping the row means validation quarantines it instead of it disappearing |
| Schema conformance | Authoritative columns in authoritative order. Extra source columns kept as `src_*` | `src_remaining_lease` (2015 onwards) is kept as a cross-check for our own remaining-lease calculation |
| Text tidy | Trim and collapse whitespace | Defensive. The current files need no fixes, but the rule is there if a future refresh does |
| Numeric cast | `floor_area_sqm`, `resale_price` → float; `lease_commence_date` → integer. Failures are listed in `_cast_errors`, never dropped | Prices can carry cents (e.g. `888888.88`), so they are floats |
| Lineage | `_source_dataset_id`, `_source_row` | Every master row can be traced back to its exact raw-file row |

**Reconciliation:** the stage checks that `len(master)` equals the sum of in-scope raw rows, and fails if it doesn't.

In [6]:
from hdb_pipeline.standardise import standardise, write_staging

master, std_summary = standardise(CFG, manifest)
std_summary.drop(columns="name")

2026-10-18 17:53:28,237 INFO    hdb_pipeline.standardise | Authoritative schema from d_43f493c6c50d54243cc1eab0df142d6a (Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012): ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
2026-10-18 17:53:29,568 INFO    hdb_pipeline.standardise | Master dataset: 113053 rows from 4 sources; whitespace fixes: none; cast failures: 0


,dataset_id,rows_raw,rows_in_scope,unparseable_month,missing_cols,extra_cols
0,d_43f493c6c50d54243cc1eab0df142d6a,369651,3188,0,[],[]
1,d_2d5ff9ea31397b66239f245f57751537,52203,52203,0,[],[]
2,d_ea9ed51da2787afaf8e51f827c304208,37153,37153,0,[],[remaining_lease]
3,d_8b84c4ee58e3cfc0ece0d773c8ca6abc,242955,20509,0,[],[remaining_lease]


In [7]:
# Reconciliation and data-quality signals from this stage
print(f"Master rows            : {len(master):,}")
print(f"Sum of in-scope raw    : {std_summary['rows_in_scope'].sum():,}")
print(f"Rows with cast errors  : {master['_cast_errors'].str.len().gt(0).sum():,}")
print(f"Whitespace fixes       : {std_summary.attrs['tidied_values']}")
master.dtypes.to_frame("dtype").T

Master rows            : 113,053
Sum of in-scope raw    : 113,053
Rows with cast errors  : 0
Whitespace fixes       : {'month': 0, 'town': 0, 'flat_type': 0, 'block': 0, 'street_name': 0, 'storey_range': 0, 'flat_model': 0, 'src_remaining_lease': 0}


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,src_remaining_lease,_cast_errors,_source_dataset_id,_source_row
dtype,string,string,string,string,string,string,Float64,string,Int64,Float64,string,object,str,int64


In [8]:
# Rows per source and the shape of src_remaining_lease, which changes format in 2017
(master.groupby("_source_dataset_id")
       .agg(rows=("month", "size"), first_month=("month", "min"), last_month=("month", "max"),
            remaining_lease_example=("src_remaining_lease", lambda s: s.dropna().iloc[0] if s.notna().any() else None))
       .sort_values("first_month"))

,rows,first_month,last_month,remaining_lease_example
_source_dataset_id,,,,
d_43f493c6c50d54243cc1eab0df142d6a,3188,2012-01,2012-02,<NA>
d_2d5ff9ea31397b66239f245f57751537,52203,2012-03,2014-12,<NA>
d_ea9ed51da2787afaf8e51f827c304208,37153,2015-01,2016-12,70
d_8b84c4ee58e3cfc0ece0d773c8ca6abc,20509,2017-01,2017-12,61 years 04 months


In [9]:
staging_path = write_staging(master, CFG)
print(f"Wrote {staging_path.relative_to(Path.cwd())}")
master.sample(5, random_state=42)

Wrote data/staging/master.parquet


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,src_remaining_lease,_cast_errors,_source_dataset_id,_source_row
49773,2014-09,BEDOK,3 ROOM,76,BEDOK NTH RD,07 TO 09,59.0,Improved,1978,295000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,46586
6491,2012-04,JURONG WEST,3 ROOM,210,BOON LAY PL,01 TO 05,65.0,Improved,1977,300000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,3304
11616,2012-06,TAMPINES,5 ROOM,734,TAMPINES ST 71,07 TO 09,123.0,Improved,1997,543000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,8429
100858,2017-06,BUKIT MERAH,4 ROOM,110,BT PURMEI RD,10 TO 12,104.0,Model A,1984,582000.0,66 years 03 months,[],d_8b84c4ee58e3cfc0ece0d773c8ca6abc,8315
106690,2017-09,JURONG WEST,4 ROOM,276B,JURONG WEST AVE 3,01 TO 03,90.0,Model A,2002,310000.0,83 years 06 months,[],d_8b84c4ee58e3cfc0ece0d773c8ca6abc,14147


## 3. Profile the master dataset

Requirement 2. These are our own profiling checks, kept small and specific to this dataset. They live in `src/hdb_pipeline/profile.py`. `profile_dataset()` is re-run later on the **Cleaned** and **Quarantined** outputs, so the report can show how the data changes through the pipeline.

| Check | Question it answers |
|---|---|
| `overview` | How many rows, from how many sources? Is any month in 2012-01 to 2017-12 missing? |
| `columns` | Type, nulls, distinct count and min/max per column |
| `domains` | Which town / flat_type / flat_model / storey_range values are **not** in the Jan 2012 reference set, how many rows have them, and when do they appear? |
| `numeric` | Spread of price, floor area and price per sqm by flat type (1st, 50th and 99th percentiles) |
| `duplicates` | Exact duplicates across the 10 business columns. Lineage columns are ignored |
| `consistency` | Values that contradict each other: lease starting after the sale year, zero or negative price or area, a block with no digits (that would break the Resale Identifier), a badly formatted storey range |

In [10]:
from hdb_pipeline.profile import profile_dataset, reference_set

reference = reference_set(master, CFG)          # Jan 2012 rows: the authoritative set
print(f"Reference set: {len(reference):,} rows from {CFG['validation']['reference_month']}")

prof_master = profile_dataset(master, reference, CFG)
prof_master["overview"]

Reference set: 1,559 rows from 2012-01


,rows,sources,first_month,last_month,months_expected,months_missing,missing_list,rows_per_month_min,rows_per_month_median,rows_per_month_max
0,113053,4,2012-01,2017-12,72,0,[],886,1563,2360


In [11]:
prof_master["columns"]

,column,dtype,nulls,distinct,min,max,top_value
0,month,string,0,72,NaN,NaN,2012-03
1,town,string,0,26,NaN,NaN,JURONG WEST
2,flat_type,string,0,7,NaN,NaN,4 ROOM
3,block,string,0,2193,NaN,NaN,2
4,street_name,string,0,526,NaN,NaN,YISHUN RING RD
5,storey_range,string,0,25,NaN,NaN,04 TO 06
6,floor_area_sqm,Float64,0,169,31.0,280.0,67.0
7,flat_model,string,0,20,NaN,NaN,Model A
8,lease_commence_date,Int64,0,50,1966.0,2015.0,1985
9,resale_price,Float64,0,2865,175000.0,1180000.0,400000.0


In [12]:
# Values outside the Jan 2012 reference set: these are the rows validation will quarantine
prof_master["domains"]

,column,value,rows,first_month,last_month
0,flat_model,DBSS,583,2014-03,2017-12
1,flat_model,Type S1,179,2014-12,2017-12
2,flat_model,Type S2,78,2015-01,2017-11
3,flat_model,Improved-Maisonette,15,2012-07,2017-10
4,flat_model,Premium Maisonette,8,2012-09,2017-09
5,flat_model,Premium Apartment Loft,6,2015-11,2017-10
6,flat_model,2-room,1,2016-10,2016-10
7,storey_range,01 TO 05,2700,2012-03,2012-05
8,storey_range,06 TO 10,2474,2012-03,2012-05
9,storey_range,11 TO 15,1259,2012-03,2012-05


In [13]:
prof_master["numeric"]

,resale_price|count,resale_price|min,resale_price|1%,resale_price|50%,resale_price|99%,resale_price|max,floor_area_sqm|count,floor_area_sqm|min,floor_area_sqm|1%,floor_area_sqm|50%,floor_area_sqm|99%,floor_area_sqm|max,price_per_sqm|count,price_per_sqm|min,price_per_sqm|1%,price_per_sqm|50%,price_per_sqm|99%,price_per_sqm|max
flat_type,,,,,,,,,,,,,,,,,,
1 ROOM,65.0,175000.0,184600.0,225000.0,258080.0,260000.0,65.0,31.0,31.0,31.0,31.0,31.0,65.0,5645.0,5955.0,7258.0,8325.0,8387.0
2 ROOM,1175.0,185000.0,199480.0,260000.0,338520.0,456888.0,1175.0,34.0,34.0,45.0,56.0,67.0,1175.0,3732.0,4241.0,5682.0,7405.0,8723.0
3 ROOM,31354.0,197000.0,242000.0,328000.0,545000.0,1150000.0,31354.0,49.0,56.0,67.0,88.0,280.0,31354.0,2791.0,3603.0,4838.0,8094.0,10582.0
4 ROOM,45139.0,247000.0,310000.0,425000.0,780000.0,998000.0,45139.0,70.0,82.0,94.0,113.0,136.0,45139.0,2375.0,3154.0,4409.0,8506.0,10645.0
5 ROOM,26370.0,330000.0,370000.0,500000.0,920000.0,1180000.0,26370.0,104.0,109.0,120.0,139.0,160.0,26370.0,2612.0,3125.0,4254.0,8048.0,10636.0
EXECUTIVE,8919.0,393000.0,455000.0,615000.0,938000.0,1160000.0,8919.0,124.0,125.0,145.0,187.0,243.0,8919.0,2912.0,3326.0,4313.0,6293.0,7452.0
MULTI-GENERATION,31.0,610000.0,622000.0,745000.0,977000.0,980000.0,31.0,132.0,133.0,164.0,177.0,179.0,31.0,3883.0,3907.0,4677.0,6013.0,6045.0


In [14]:
display(prof_master["duplicates"])
prof_master["consistency"]

,duplicate_rows
d_2d5ff9ea31397b66239f245f57751537,248
d_43f493c6c50d54243cc1eab0df142d6a,1
d_8b84c4ee58e3cfc0ece0d773c8ca6abc,178
d_ea9ed51da2787afaf8e51f827c304208,24
ALL,451


,check,rows_flagged
0,lease_commence_after_sale_year,0
1,non_positive_price,0
2,non_positive_floor_area,0
3,block_without_digits,0
4,storey_range_bad_format,0


In [15]:
# The extreme 3 ROOM floor area found in the numeric profile
master[(master["flat_type"] == "3 ROOM") & (master["floor_area_sqm"] > 150)]

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,src_remaining_lease,_cast_errors,_source_dataset_id,_source_row
15301,2012-08,KALLANG/WHAMPOA,3 ROOM,65,JLN MA'MOR,01 TO 03,249.0,Terrace,1972,988888.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,12114
27924,2013-04,KALLANG/WHAMPOA,3 ROOM,65,JLN MA'MOR,01 TO 03,266.0,Terrace,1972,1020000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,24737
37742,2013-11,KALLANG/WHAMPOA,3 ROOM,59,JLN MA'MOR,01 TO 03,180.0,Terrace,1972,993888.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,34555
58504,2015-03,KALLANG/WHAMPOA,3 ROOM,53,JLN MA'MOR,01 TO 03,280.0,Terrace,1972,1060000.0,56,[],d_ea9ed51da2787afaf8e51f827c304208,3114
63296,2015-06,KALLANG/WHAMPOA,3 ROOM,60,JLN BAHAGIA,01 TO 03,241.0,Terrace,1972,958000.0,56,[],d_ea9ed51da2787afaf8e51f827c304208,7906
91881,2016-12,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.0,Terrace,1972,1150000.0,54,[],d_ea9ed51da2787afaf8e51f827c304208,36491
99544,2017-05,KALLANG/WHAMPOA,3 ROOM,59,JLN MA'MOR,01 TO 03,181.0,Terrace,1972,810000.0,54 years 02 months,[],d_8b84c4ee58e3cfc0ece0d773c8ca6abc,7001
101412,2017-06,KALLANG/WHAMPOA,3 ROOM,38,JLN BAHAGIA,01 TO 03,215.0,Terrace,1972,830000.0,54 years 01 month,[],d_8b84c4ee58e3cfc0ece0d773c8ca6abc,8869
112237,2017-12,KALLANG/WHAMPOA,3 ROOM,65,JLN MA'MOR,01 TO 03,249.0,Terrace,1972,1053888.0,53 years 07 months,[],d_8b84c4ee58e3cfc0ece0d773c8ca6abc,19694


### Observations

- **Complete coverage.** All 72 months from 2012-01 to 2017-12 are present (between 886 and 2,360 sales a month). No nulls, and every value converts to its type.
- **town and flat_type match Jan 2012 exactly**: 26 towns and 7 flat types, with no new values.
- **flat_model drift**: 7 values never appear in Jan 2012. DBSS (583 rows) and Type S1/S2 (257 rows) are newer products that first resold from 2014. Strict validation will quarantine them, so analysis of the cleaned data won't include these newer flat types.
- **storey_range drift has two causes:**
  1. **5-floor bands** (`01 TO 05`, `06 TO 10`, …) appear **only in Mar–May 2012**, about 6,800 rows. This is a change in how floors were recorded at the switch from approval-date to registration-date files, not bad data. It can't be mapped onto 3-floor bands because the bands don't line up.
  2. **Floors above 36** (`37 TO 39` up to `49 TO 51`, about 210 rows) are genuine high-rise flats that just didn't trade in Jan 2012.
- **Duplicates**: 451 exact duplicate records, mostly within the same file. Some may be genuinely separate sales of similar flats, since the data has no unit number. Following the brief, the first copy is kept and the rest are quarantined.
- **No consistency violations.** Every block contains digits, which matters for the Resale Identifier step.
- **An outlier that isn't an anomaly**: nine 3 ROOM flats are 180–280 sqm (the 99th percentile is 88 sqm) and sell for about $1M. All of them are HDB **Terrace** houses on Jalan Ma'mor and Jalan Bahagia (Kallang/Whampoa, lease from 1972). They're genuine. The lesson for anomaly detection: compare prices within peer groups that include `flat_model`, and use price per sqm, not just price by flat type. Otherwise rare but genuine housing types get flagged.
- **`src_remaining_lease` format is inconsistent within 2017**: `54 years 02 months` sits alongside `54 years 01 month` (singular). The cross-check parser has to handle both.

## 4. Validate → Cleaned / Quarantined

Requirement 3, plus the extra rules allowed by requirement 6. Every rule is a function that returns a True/False mask (True = fails), keyed by a **reason code**. A row that fails several rules records **all** of them in `_dq_reasons`, joined with `|`.

| Reason code | Rule |
|---|---|
| `CAST_ERROR` | A numeric value couldn't be cast during standardisation |
| `INVALID_MONTH` | `month` isn't in `YYYY-MM` format |
| `MISSING_VALUE` | A business column is null or empty |
| `INVALID_TOWN` / `_FLAT_TYPE` / `_FLAT_MODEL` / `_STOREY_RANGE` | The value isn't in the **Jan 2012 authoritative set**. The comparison ignores case, and passing values are rewritten to the reference spelling |
| `NON_POSITIVE_PRICE` / `NON_POSITIVE_AREA` | Price or floor area ≤ 0 |
| `LEASE_AFTER_SALE` | The lease starts after the sale year |
| `DUPLICATE` | Exact copy of an earlier record across the 10 business columns. The first copy, ordered by source then source row, stays in Cleaned |

**Design choices**
- **Quarantine, don't drop.** Every rejected row keeps its lineage and reasons, so it can be fixed and reprocessed later.
- **Strict means strict on values, not formatting.** Capitalisation doesn't fail a row. Different words do.
- **The duplicate pick is deterministic**, so reruns always keep the same copy.

**Reconciliation:** `cleaned + quarantined == master`, with no row in both. The stage checks this itself.

In [16]:
from hdb_pipeline.validate import validate
from hdb_pipeline.io import write_output

cleaned, quarantined, rule_summary = validate(master, reference, CFG)

print(f"Master      : {len(master):>8,}")
print(f"Cleaned     : {len(cleaned):>8,}  ({len(cleaned)/len(master):.1%})")
print(f"Quarantined : {len(quarantined):>8,}  ({len(quarantined)/len(master):.1%})")
rule_summary  # rows_failed counts every failure; only_reason = rows this rule alone sent to quarantine

2026-10-18 17:53:34,201 INFO    hdb_pipeline.validate | Validation: 104783 cleaned, 8270 quarantined (7.3%)


Master      :  113,053
Cleaned     :  104,783  (92.7%)
Quarantined :    8,270  (7.3%)


,rule,rows_failed,only_reason,pct_of_master
0,CAST_ERROR,0,0,0.00
1,INVALID_MONTH,0,0,0.00
2,MISSING_VALUE,0,0,0.00
3,INVALID_TOWN,0,0,0.00
4,INVALID_FLAT_TYPE,0,0,0.00
5,INVALID_FLAT_MODEL,870,775,0.77
6,INVALID_STOREY_RANGE,7050,6952,6.24
7,NON_POSITIVE_PRICE,0,0,0.00
8,NON_POSITIVE_AREA,0,0,0.00
9,LEASE_AFTER_SALE,0,0,0.00


In [17]:
# Reason combinations, and where quarantined rows come from
display(quarantined["_dq_reasons"].value_counts().rename("rows").to_frame())
pd.crosstab(quarantined["_source_dataset_id"], quarantined["_dq_reasons"], margins=True)

,rows
_dq_reasons,
INVALID_STOREY_RANGE,6952
INVALID_FLAT_MODEL,775
DUPLICATE,442
INVALID_FLAT_MODEL|INVALID_STOREY_RANGE,92
INVALID_STOREY_RANGE|DUPLICATE,6
INVALID_FLAT_MODEL|DUPLICATE,3


_dq_reasons,DUPLICATE,INVALID_FLAT_MODEL,INVALID_FLAT_MODEL|DUPLICATE,INVALID_FLAT_MODEL|INVALID_STOREY_RANGE,INVALID_STOREY_RANGE,INVALID_STOREY_RANGE|DUPLICATE,All
_source_dataset_id,,,,,,,
d_2d5ff9ea31397b66239f245f57751537,242,83,0,0,6869,6,7200
d_43f493c6c50d54243cc1eab0df142d6a,1,0,0,0,1,0,2
d_8b84c4ee58e3cfc0ece0d773c8ca6abc,176,340,2,36,39,0,593
d_ea9ed51da2787afaf8e51f827c304208,23,352,1,56,43,0,475
All,442,775,3,92,6952,6,8270


In [18]:
# Quarantined rows over time: the storey-range failures cluster in Mar–May 2012
q_by_month = pd.crosstab(quarantined["month"], quarantined["_dq_reasons"].str.split("|").str[0])
q_by_month[q_by_month.sum(axis=1) > 100]

_dq_reasons,DUPLICATE,INVALID_FLAT_MODEL,INVALID_STOREY_RANGE
month,,,
2012-03,0,0,2360
2012-04,0,0,2155
2012-05,0,0,2323
2014-05,122,7,2


In [19]:
paths = write_output(quarantined, "quarantined", CFG)
print({k: str(v.relative_to(Path.cwd())) for k, v in paths.items()})
quarantined.sample(5, random_state=1)

{'parquet': 'data/quarantined/quarantined.parquet', 'csv': 'data/quarantined/quarantined.csv'}


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,src_remaining_lease,_cast_errors,_source_dataset_id,_source_row,_dq_reasons
9636,2012-05,TOA PAYOH,4 ROOM,122,LOR 2 TOA PAYOH,01 TO 05,100.0,Model A,1999,580000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,6449,INVALID_STOREY_RANGE
3632,2012-03,BUKIT MERAH,4 ROOM,102,HENDERSON CRES,11 TO 15,80.0,Standard,1970,485000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,445,INVALID_STOREY_RANGE
3939,2012-03,CLEMENTI,4 ROOM,302,CLEMENTI AVE 4,01 TO 05,92.0,New Generation,1979,495000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,752,INVALID_STOREY_RANGE
9438,2012-05,TAMPINES,3 ROOM,252,TAMPINES ST 21,01 TO 05,72.0,New Generation,1985,348000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,6251,INVALID_STOREY_RANGE
4936,2012-03,TAMPINES,3 ROOM,206,TAMPINES ST 21,06 TO 10,79.0,Model A,1982,370000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,1749,INVALID_STOREY_RANGE


### 4a. Re-profile after validation

The same `profile_dataset()` from Section 3, run on both outputs. Cleaned should show **no domain drift** and **no duplicates**. Quarantined should contain only the issues found in Section 3.

In [20]:
prof_cleaned = profile_dataset(cleaned, reference, CFG)
prof_quarantined = profile_dataset(quarantined, reference, CFG)

lifecycle = pd.DataFrame({
    stage: {
        "rows": len(df),
        "domain_drift_rows": int(p["domains"]["rows"].sum()),
        "duplicate_rows": int(p["duplicates"].loc["ALL", "duplicate_rows"]),
        "months_missing": int(p["overview"]["months_missing"].iloc[0]),
        "distinct_flat_model": int(p["columns"].set_index("column").loc["flat_model", "distinct"]),
        "distinct_storey_range": int(p["columns"].set_index("column").loc["storey_range", "distinct"]),
    }
    for stage, df, p in [("master", master, prof_master), ("cleaned", cleaned, prof_cleaned),
                         ("quarantined", quarantined, prof_quarantined)]
}).T
lifecycle

,rows,domain_drift_rows,duplicate_rows,months_missing,distinct_flat_model,distinct_storey_range
master,113053,7920,451,0,20,25
cleaned,104783,0,0,3,13,12
quarantined,8270,7920,9,3,20,25


In [21]:
# Does cleaning skew the data? Compare the cleaned and master mix by flat type
mix = pd.DataFrame({"master": master["flat_type"].value_counts(normalize=True),
                    "cleaned": cleaned["flat_type"].value_counts(normalize=True)}).mul(100).round(2)
mix["change_pct_pts"] = (mix["cleaned"] - mix["master"]).round(2)
mix

,master,cleaned,change_pct_pts
flat_type,,,
4 ROOM,39.93,39.96,0.03
3 ROOM,27.73,27.95,0.22
5 ROOM,23.33,23.01,-0.32
EXECUTIVE,7.89,7.94,0.05
2 ROOM,1.04,1.05,0.01
1 ROOM,0.06,0.06,0.0
MULTI-GENERATION,0.03,0.02,-0.01


In [22]:
from hdb_pipeline.profile import BUSINESS_COLS

# Duplicate spike: copies on consecutive raw rows point to a double-load, not separate identical sales
dups = quarantined[quarantined["_dq_reasons"].str.contains("DUPLICATE")]
display(dups.groupby("month").size().sort_values(ascending=False).head(5).rename("duplicate_rows").to_frame())

may14 = master[master["month"] == "2014-05"]
groups = may14.groupby(BUSINESS_COLS, dropna=False)["_source_row"].apply(list)
pairs = groups[groups.str.len() > 1]
print(f"2014-05: {len(pairs)} duplicate groups; copies on consecutive raw rows: "
      f"{sum(r[1] - r[0] == 1 for r in pairs)}, within 2 rows: {sum(r[1] - r[0] <= 2 for r in pairs)}")
pairs.head()

,duplicate_rows
month,
2014-05,122
2014-04,65
2017-10,20
2017-08,20
2017-05,20


2014-05: 122 duplicate groups; copies on consecutive raw rows: 97, within 2 rows: 114


month    town        flat_type  block  street_name        storey_range  floor_area_sqm  flat_model      lease_commence_date  resale_price
2014-05  ANG MO KIO  3 ROOM     126    ANG MO KIO AVE 3   04 TO 06      82.0            New Generation  1978                 401888.0        [41209, 41210]
                                302    ANG MO KIO AVE 3   07 TO 09      73.0            New Generation  1978                 375000.0        [41196, 41198]
                                417    ANG MO KIO AVE 10  07 TO 09      74.0            New Generation  1979                 368000.0        [41193, 41194]
                                533    ANG MO KIO AVE 5   10 TO 12      82.0            New Generation  1980                 390000.0        [41203, 41205]
                                556    ANG MO KIO AVE 10  01 TO 03      68.0            New Generation  1980                 300000.0        [41169, 41170]
Name: _source_row, dtype: object

### Observations

- **7.3% quarantined** (8,270 of 113,053). Almost all of it comes from two rules: `INVALID_STOREY_RANGE` (7,050) and `INVALID_FLAT_MODEL` (870). Only 101 rows fail more than one rule.
- **⚠️ Cleaned has a 3-month gap: Mar–May 2012 are 100% quarantined.** Every sale in those months was recorded with 5-floor bands (`01 TO 05`, …), which aren't in the Jan 2012 set. This follows the strict rule the brief requires, but **the Data Science Team must treat Mar–May 2012 as missing** when building time series or month-on-month comparisons. The rows are kept in Quarantined in full, so a looser rule could restore them later. A 5-floor band can't be mapped onto 3-floor bands without guessing the floor.
- **Duplicates spike in Apr–May 2014** (65 and 122, against a median of 2 a month; the next-highest months reach about 20), and the copies sit on consecutive raw-file rows. That looks like a double-load at the source, which supports quarantining duplicates rather than treating them as separate sales. Duplicates in other months are more likely genuine separate sales of similar flats, since the data has no unit number. These are kept in quarantine too, under the same rule.
- **Quarantining doesn't skew the flat-type mix**: no flat type's share moves by more than 0.4 percentage points between master and cleaned.
- **Cleaned is internally consistent**: no domain drift and no duplicates, and its `flat_model` and `storey_range` values match the Jan 2012 set exactly (13 and 12).

## 5. Enrich → Cleaned

### 5a. Remaining lease as of 1 Jan 2027 (requirement 4)

`lease_commence_date` is a **year only**. **Assumption:** every lease starts on 1 Jan of that year and runs 99 years:

```
months_left = (lease_commence_date + 99) × 12 − 2027 × 12      (never below 0)
years, months = divmod(months_left, 12)                         (rounded down)
```

Because the assumed start is always 1 Jan, **`months` is always 0**. The code still works in months, so it would give correct months if a full start date ever became available.

**Cross-check:** HDB publishes its own `remaining_lease` from 2015 onwards, calculated from the actual start date at the time of sale. Applying our method at each sale date and comparing the two shows how much the year-only assumption costs.

In [23]:
from hdb_pipeline.enrich import enrich, lease_cross_check

cleaned = enrich(cleaned, CFG)

display(cleaned["remaining_lease_years"].describe().round(1).to_frame().T)
print("Distinct remaining_lease_months values:", cleaned["remaining_lease_months"].unique().tolist())
cleaned[["lease_commence_date", "remaining_lease_years", "remaining_lease_months", "remaining_lease_2027"]].drop_duplicates().sort_values("lease_commence_date").head()

,count,mean,std,min,25%,50%,75%,max
remaining_lease_years,104783.0,62.4,10.7,38.0,55.0,60.0,71.0,87.0


Distinct remaining_lease_months values: [0]


,lease_commence_date,remaining_lease_years,remaining_lease_months,remaining_lease_2027
23803,1966,38,0,38 years 00 months
10375,1967,39,0,39 years 00 months
10698,1968,40,0,40 years 00 months
10372,1969,41,0,41 years 00 months
10394,1970,42,0,42 years 00 months


In [24]:
# Ours minus HDB's, in months. Negative = our figure is shorter
lease_cross_check(cleaned, CFG)

,rows,diff_months_mean,diff_months_std,diff_months_min,diff_months_5%,diff_months_50%,diff_months_95%,diff_months_max
era,,,,,,,,
2015-16 (years),36678.0,0.1,5.0,-13.0,-8.0,0.0,8.0,11.0
2017 (years+months),19916.0,-5.4,3.5,-14.0,-11.0,-6.0,0.0,9.0


### 5b. Price anomalies (requirement 5)

Anomalies are **flagged, not quarantined**. An unusual price may well be genuine, and the Data Science Team should decide what to do with it.

| Rule | Logic |
|---|---|
| `PEER_OUTLIER` | Price per sqm is far from its peers by robust z-score: `z = 0.6745 × (x − median) / MAD`, flagged when `|z| > 3.5` (Iglewicz & Hoaglin). Median and MAD aren't distorted by the outliers being looked for, unlike mean and standard deviation |
| `PRICE_OUT_OF_RANGE` | Hard plausibility bounds on price and price per sqm (set in config) |

**Peer groups** use the most specific level with ≥ 20 sales: `town × flat_type × flat_model × lease_decade × year`, then `town × flat_type × flat_model × year`, then `flat_type × flat_model × year`, then `flat_type × flat_model`, then `flat_type`.

How the levels were chosen (tried on this data):
- An early version fell back to `flat_type` alone and **flagged 57 of 70 Terrace houses**. They're genuine, but very large for 3 ROOM flats, so they look cheap per sqm. **Rule: never drop `flat_model`.**
- Dropping `town` before `lease_decade` flagged premium central locations (Toa Payoh, Queenstown). Location matters more than lease age, so `lease_decade` is dropped first.

In [25]:
flags = cleaned["anomaly_reasons"].replace("", "(none)").value_counts().rename("rows").to_frame()
flags["pct"] = (100 * flags["rows"] / len(cleaned)).round(2)
display(flags)
display(cleaned["peer_level"].value_counts().rename("rows_scored_at_level").to_frame())
print(f"Flagged above peers: {(cleaned['is_price_anomaly'] & (cleaned['robust_z'] > 0)).sum():,} | "
      f"below peers: {(cleaned['is_price_anomaly'] & (cleaned['robust_z'] < 0)).sum():,} | "
      f"Terrace houses flagged: {cleaned.loc[cleaned['flat_model'] == 'Terrace', 'is_price_anomaly'].sum()} of {(cleaned['flat_model'] == 'Terrace').sum()}")

,rows,pct
anomaly_reasons,,
(none),103350,98.63
PEER_OUTLIER,1433,1.37


,rows_scored_at_level
peer_level,
town×flat_type×flat_model×lease_decade×sale_year,89309
town×flat_type×flat_model×sale_year,9248
flat_type×flat_model×sale_year,5822
flat_type×flat_model,398
flat_type,6


Flagged above peers: 1,210 | below peers: 223 | Terrace houses flagged: 4 of 70


In [26]:
# The most extreme flags: what does 'unusual' look like here?
show = ["month", "town", "flat_type", "flat_model", "lease_commence_date", "block", "street_name", "storey_range",
        "floor_area_sqm", "resale_price", "price_per_sqm", "peer_median_ppsm", "robust_z", "peer_level"]
anom = cleaned[cleaned["is_price_anomaly"]]
anom.reindex(anom["robust_z"].abs().sort_values(ascending=False).index)[show].head(10)

,month,town,flat_type,flat_model,lease_commence_date,block,street_name,storey_range,floor_area_sqm,resale_price,price_per_sqm,peer_median_ppsm,robust_z,peer_level
68684,2015-10,BEDOK,3 ROOM,Model A,2010,219A,BEDOK CTRL,07 TO 09,62.0,478000.0,7709.68,4150.00,17.55,town×flat_type×flat_model×sale_year
65711,2015-08,BEDOK,3 ROOM,Model A,2010,219A,BEDOK CTRL,07 TO 09,62.0,470000.0,7580.65,4150.00,16.91,town×flat_type×flat_model×sale_year
55494,2015-01,BEDOK,3 ROOM,Model A,2010,219A,BEDOK CTRL,16 TO 18,69.0,500000.0,7246.38,4150.00,15.26,town×flat_type×flat_model×sale_year
101863,2017-06,SERANGOON,4 ROOM,Model A,1999,257,SERANGOON CTRL DR,13 TO 15,99.0,680000.0,6868.69,3800.00,14.78,town×flat_type×flat_model×lease_decade×sale_year
60893,2015-05,BEDOK,3 ROOM,Model A,2010,219D,BEDOK CTRL,16 TO 18,69.0,490000.0,7101.45,4150.00,14.55,town×flat_type×flat_model×sale_year
27867,2013-04,JURONG WEST,5 ROOM,Improved,1982,522,JURONG WEST ST 52,19 TO 21,118.0,626800.0,5311.86,3617.26,13.89,town×flat_type×flat_model×lease_decade×sale_year
70405,2015-11,BEDOK,3 ROOM,Model A,2010,219D,BEDOK CTRL,04 TO 06,69.0,473000.0,6855.07,4150.00,13.33,town×flat_type×flat_model×sale_year
55493,2015-01,BEDOK,3 ROOM,Model A,2010,221B,BEDOK CTRL,04 TO 06,69.0,458000.0,6637.68,4150.00,12.26,town×flat_type×flat_model×sale_year
67180,2015-09,BEDOK,4 ROOM,Model A,1989,112,LENGKONG TIGA,07 TO 09,104.0,605000.0,5817.31,4086.54,12.14,town×flat_type×flat_model×lease_decade×sale_year
92224,2016-12,SERANGOON,4 ROOM,Model A,1999,257,SERANGOON CTRL DR,13 TO 15,99.0,680000.0,6868.69,4038.74,11.84,town×flat_type×flat_model×lease_decade×sale_year


In [27]:
# Where do flags concentrate? Repeated streets point to block-level effects (newer or taller blocks, location within a town)
anom.groupby(["town", "street_name"]).size().sort_values(ascending=False).head(8).rename("flagged_rows").to_frame()

flagged_rows
town        street_name                    
JURONG WEST JURONG WEST ST 52            63
HOUGANG     HOUGANG CTRL                 46
SERANGOON   SERANGOON CTRL DR            45
BUKIT TIMAH TOH YI DR                    43
BEDOK       LENGKONG TIGA                35
            BEDOK CTRL                   34
BISHAN      SHUNFU RD                    32
BUKIT BATOK BT BATOK CTRL                31

### 5c. Proof that the mechanism works: injected anomalies

The real data may contain no truly bad prices, so we inject some obvious ones into a **copy** of Cleaned and check that both rules catch them. The real outputs aren't touched.

In [28]:
from hdb_pipeline.enrich import price_anomalies

probe = cleaned.sample(4, random_state=7).copy()
probe["case"] = ["price x3 (typo: extra digit?)", "price / 10 (missing digit)",
                 "price = $1 (placeholder)", "unchanged (control)"]
probe.iloc[0, probe.columns.get_loc("resale_price")] *= 3
probe.iloc[1, probe.columns.get_loc("resale_price")] /= 10
probe.iloc[2, probe.columns.get_loc("resale_price")] = 1.0

base_cols = [c for c in cleaned.columns if c not in ("price_per_sqm", "peer_level", "peer_median_ppsm",
                                                    "robust_z", "anomaly_reasons", "is_price_anomaly")]
test = pd.concat([cleaned[base_cols].drop(index=probe.index), probe[base_cols]])
result = price_anomalies(test, CFG).loc[probe.index]
result.assign(case=probe["case"])[["case", "town", "flat_type", "resale_price", "price_per_sqm",
                                   "peer_median_ppsm", "robust_z", "anomaly_reasons"]]

,case,town,flat_type,resale_price,price_per_sqm,peer_median_ppsm,robust_z,anomaly_reasons
112354,price x3 (typo: extra digit?),PUNGGOL,4 ROOM,1215000.0,13206.52,4641.730,39.27,PEER_OUTLIER
60121,price / 10 (missing digit),PUNGGOL,4 ROOM,43300.0,460.64,4484.045,-11.51,PEER_OUTLIER|PRICE_OUT_OF_RANGE
96396,price = $1 (placeholder),TAMPINES,EXECUTIVE,1.0,0.01,4344.590,-12.47,PEER_OUTLIER|PRICE_OUT_OF_RANGE
23044,unchanged (control),WOODLANDS,5 ROOM,446000.0,4054.55,3891.670,0.69,


In [29]:
paths = write_output(cleaned, "cleaned", CFG)
print({k: str(v.relative_to(Path.cwd())) for k, v in paths.items()})
print(f"Cleaned: {len(cleaned):,} rows × {cleaned.shape[1]} columns")
cleaned.head(3)

{'parquet': 'data/cleaned/cleaned.parquet', 'csv': 'data/cleaned/cleaned.csv'}
Cleaned: 104,783 rows × 23 columns


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,src_remaining_lease,_cast_errors,_source_dataset_id,_source_row,remaining_lease_years,remaining_lease_months,remaining_lease_2027,price_per_sqm,peer_level,peer_median_ppsm,robust_z,anomaly_reasons,is_price_anomaly
10026,2012-06,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,04 TO 06,45.0,Improved,1986,259000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,6839,58,0,58 years 00 months,5755.56,town×flat_type×flat_model×sale_year,6022.73,-1.01,,False
10027,2012-06,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,265000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,6840,51,0,51 years 00 months,6022.73,town×flat_type×flat_model×sale_year,6022.73,0.00,,False
10028,2012-06,ANG MO KIO,2 ROOM,508,ANG MO KIO AVE 8,04 TO 06,44.0,Improved,1980,266000.0,<NA>,[],d_2d5ff9ea31397b66239f245f57751537,6841,52,0,52 years 00 months,6045.45,town×flat_type×flat_model×sale_year,6022.73,0.09,,False


### Observations

- **Remaining lease at 1 Jan 2027** ranges from 38 to 87 years (median 60). Months are always 0 because only the lease start year is known.
- **The cross-check confirms the assumption is reasonable.** For 2015–16, which HDB publishes in whole years, the median difference is 0. For 2017, which HDB publishes in years and months, our figure is a median **6 months shorter** (90% of rows within 0 to −11 months). That's what assuming a 1 Jan start for leases that really start partway through the year should produce. To be exact to the month, the data would need the lease start month.
- **1.4% of cleaned rows are flagged as price anomalies**, mostly *above* their peers. The flags cluster on particular streets (e.g. newer or taller blocks in an otherwise older town), which suggests block-level premiums the peer groups can't see, such as a new block, a high floor, a view or the spot within a town. They're review candidates, not errors.
- **No rows break the hard price bounds.** The injected test shows both rules catching a ×3 typo, a missing digit and a $1 placeholder, and leaving the unchanged control row alone.

## 6. Transform → Transformed (Resale Identifier)

Requirement 8, built on the **Cleaned** (validated) data.

**Base identifier: 12 characters, per the spec**

| Part | Rule | Example |
|---|---|---|
| `S` | Literal | `S` |
| `BBB` | Digits of `block` only, first 3, left-padded with zeros | `19` → `019`, `436B` → `436` |
| `PPP` | First 3 digits of the **average** `resale_price` over (year-month, town, flat_type), truncated to whole dollars | $230,499.50 → `230` |
| `YYMM` | Year-month of the sale | 2012-01 → `1201` |
| `T` | First letter of town | ANG MO KIO → `A` |

**Making it unique without dropping records.** Every row gets a 3-digit sequence number within its base ID: `resale_identifier = base + NNN`, e.g. `S0192301201A001`.
- **Deterministic:** within each base ID, rows are numbered in a fixed order based on the data (storey range, floor area, price, lease, model, street, then source row), so reruns give identical IDs whatever order the rows arrive in.
- **Fixed length (15 characters) for every row**, whether or not it collided. Downstream systems never have to handle two ID formats, and the spec's base ID is always the first 12 characters.
- **Fails loudly** if a base group ever outgrows 999 rows, or if the final IDs aren't unique.

The parts of the ID (`rid_*` columns) are kept in Transformed so every ID can be traced back to how it was built.

In [30]:
from hdb_pipeline.transform import transform

transformed, id_summary = transform(cleaned, CFG)
id_summary.T.rename(columns={0: "value"})

2026-10-18 17:53:43,811 INFO    hdb_pipeline.transform | Resale Identifier: 104783 rows, 91831 base IDs, 104783 final IDs (unique)


,value
rows,104783.00
distinct_base_ids,91831.00
rows_with_colliding_base_id,23639.00
pct_rows_colliding,22.56
largest_collision_group,14.00
distinct_final_ids,104783.00


In [31]:
# How big are the collision groups, and what causes them?
sizes = transformed.groupby("rid_base").size()
display(sizes.value_counts().sort_index().rename_axis("rows_sharing_a_base_id").rename("base_ids").to_frame().T)

largest = sizes.idxmax()
transformed[transformed["rid_base"] == largest].sort_values("rid_sequence")[
    ["month", "town", "flat_type", "block", "street_name", "storey_range", "floor_area_sqm",
     "resale_price", "rid_avg_price", "rid_base", "resale_identifier"]]

rows_sharing_a_base_id,1,2,3,4,5,6,7,8,9,14
base_ids,81144,9004,1305,252,80,28,12,3,2,1


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,resale_price,rid_avg_price,rid_base,resale_identifier
28155,2013-04,SENGKANG,4 ROOM,436B,FERNVALE RD,04 TO 06,96.0,519000.0,515574.7,S4365151304S,S4365151304S001
28149,2013-04,SENGKANG,4 ROOM,436B,FERNVALE RD,07 TO 09,96.0,505000.0,515574.7,S4365151304S,S4365151304S002
28153,2013-04,SENGKANG,4 ROOM,436B,FERNVALE RD,07 TO 09,96.0,518000.0,515574.7,S4365151304S,S4365151304S003
28158,2013-04,SENGKANG,4 ROOM,436A,FERNVALE RD,07 TO 09,96.0,525000.0,515574.7,S4365151304S,S4365151304S004
28162,2013-04,SENGKANG,4 ROOM,436A,FERNVALE RD,10 TO 12,96.0,530000.0,515574.7,S4365151304S,S4365151304S005
28176,2013-04,SENGKANG,4 ROOM,436C,FERNVALE RD,13 TO 15,96.0,545000.0,515574.7,S4365151304S,S4365151304S006
28172,2013-04,SENGKANG,4 ROOM,436C,FERNVALE RD,16 TO 18,96.0,536000.0,515574.7,S4365151304S,S4365151304S007
28157,2013-04,SENGKANG,4 ROOM,436D,FERNVALE RD,19 TO 21,96.0,525000.0,515574.7,S4365151304S,S4365151304S008
28167,2013-04,SENGKANG,4 ROOM,436C,FERNVALE RD,19 TO 21,96.0,535000.0,515574.7,S4365151304S,S4365151304S009
28168,2013-04,SENGKANG,4 ROOM,436A,FERNVALE RD,19 TO 21,96.0,535000.0,515574.7,S4365151304S,S4365151304S010


In [32]:
# Edge cases in the spec rules, measured on this data
blk_digits = transformed["block"].str.replace(r"\D", "", regex=True)
edge_cases = {
    "Blocks with letters stripped (e.g. 436A/436B -> 436)": transformed["block"].str.contains(r"\D").sum(),
    "Blocks with more than 3 digits (truncated)": blk_digits.str.len().gt(3).sum(),
    "Blocks with fewer than 3 digits (zero-padded)": blk_digits.str.len().lt(3).sum(),
    "Rows whose group average is >= $1M (7-digit average)": (transformed["rid_avg_price"] >= 1_000_000).sum(),
    "Final IDs unique": transformed["resale_identifier"].is_unique,
}
pd.Series(edge_cases, name="value").to_frame()

,value
Blocks with letters stripped (e.g. 436A/436B -> 436),20432
Blocks with more than 3 digits (truncated),0
Blocks with fewer than 3 digits (zero-padded),16160
Rows whose group average is >= $1M (7-digit average),5
Final IDs unique,True


In [33]:
paths = write_output(transformed, "transformed", CFG)
print({k: str(v.relative_to(Path.cwd())) for k, v in paths.items()})
transformed[["month", "town", "flat_type", "block", "resale_price", "rid_avg_price", "rid_base", "resale_identifier"]].head()

{'parquet': 'data/transformed/transformed.parquet', 'csv': 'data/transformed/transformed.csv'}


,month,town,flat_type,block,resale_price,rid_avg_price,rid_base,resale_identifier
10026,2012-06,ANG MO KIO,2 ROOM,172,259000.0,266500.0,S1722661206A,S1722661206A001
10027,2012-06,ANG MO KIO,2 ROOM,406,265000.0,266500.0,S4062661206A,S4062661206A001
10028,2012-06,ANG MO KIO,2 ROOM,508,266000.0,266500.0,S5082661206A,S5082661206A001
10029,2012-06,ANG MO KIO,2 ROOM,323,270000.0,266500.0,S3232661206A,S3232661206A001
10030,2012-06,ANG MO KIO,2 ROOM,314,272500.0,266500.0,S3142661206A,S3142661206A001


### Observations

- **22.6% of rows (23,639) share a base ID with at least one other row.** Largest group: 14 sales. The 15-character final IDs are all unique (104,783 distinct), and no rows are dropped.
- **Why so many collisions:** the spec's rule **removes block letters**, so `436A`, `436B` and `436C FERNVALE RD` all become `436`. About 20,000 cleaned rows have lettered blocks. And since every sale in one month, town and flat type shares the same average, the price part adds nothing to tell those sales apart.
- **One ambiguity in the spec:** an average of $1M or more gives a 7-digit number, so its first 3 digits (`100`–`118`) look like an average of about $100k. This affects 5 rows here, and no real average in this data is below $100k, so they can't be confused in practice. Still, the price part is a *prefix*, not a scaled value, and anyone using the ID should know that.
- **Limitation:** the sequence number is stable for this fixed 2012–2017 history. If late registrations were later added to a past month, the numbering in that group could shift. A production system would assign IDs once and store them, rather than recompute them.

## 7. De-identify → Hashed

Requirement 9: replace the identifier with a hash that is **irreversible**, **keeps uniqueness** and **resists dictionary attacks**.

| Option | Irreversible | Unique | Resists dictionary attack | Deterministic (joinable, reproducible) |
|---|---|---|---|---|
| Plain SHA-256 | ✅ | ✅ | ❌ only a few million possible IDs, so an attacker can hash them all and match | ✅ |
| SHA-256 + random salt per row | ✅ | ✅ | ✅ | ❌ the same ID hashes differently each run |
| **HMAC-SHA256 + secret key** ✔ | ✅ | ✅ | ✅ an attacker can't build the lookup table without the key | ✅ |

**Key handling:** the key is read from the `HDB_HASH_KEY` environment variable and is never stored in code, config or Git. If it isn't set, a clearly labelled **demo key** is used and a warning is logged, so reviewers can still run the notebook. In production the key would live in AWS Secrets Manager, and `allow_demo_key` would be set to `false`.

**Hashed output** = Cleaned columns + `resale_identifier_hash`. The plain identifier **and its `rid_*` parts are removed**, because the parts alone would let someone rebuild it.

In [34]:
from hdb_pipeline.hashing import hash_identifier

hashed, key_source = hash_identifier(transformed, CFG)
print(f"Key source      : {key_source}{'  <- DEMO KEY: set HDB_HASH_KEY for real use' if key_source == 'demo' else ''}")
print(f"Rows            : {len(hashed):,}")
print(f"Unique hashes   : {hashed['resale_identifier_hash'].nunique():,}")
print(f"Plain ID present: {'resale_identifier' in hashed.columns or any(c.startswith('rid_') for c in hashed.columns)}")
hashed[["month", "town", "flat_type", "block", "resale_price", "resale_identifier_hash"]].head()

2026-10-18 17:53:45,227 WARNING hdb_pipeline.hashing | HDB_HASH_KEY not set: using the DEMO key. Output hashes are NOT protected.


Key source      : demo  <- DEMO KEY: set HDB_HASH_KEY for real use
Rows            : 104,783
Unique hashes   : 104,783
Plain ID present: False


,month,town,flat_type,block,resale_price,resale_identifier_hash
10026,2012-06,ANG MO KIO,2 ROOM,172,259000.0,aef8cd0cbdece66dfe5ed6d4bb47586310d44ca1508d52...
10027,2012-06,ANG MO KIO,2 ROOM,406,265000.0,bb8bd7bcda74d8ff755f220e0aea67e385c6ccc5c7a616...
10028,2012-06,ANG MO KIO,2 ROOM,508,266000.0,b31e52d375adf3c1cd386824fea0a59f9244d5fa26e34f...
10029,2012-06,ANG MO KIO,2 ROOM,323,270000.0,fe4333e98451f2b25846b6c03b141af0786753ae3ce618...
10030,2012-06,ANG MO KIO,2 ROOM,314,272500.0,903879447d7bf8f7a19311b47c68dc63e39809f35eb670...


In [35]:
# Dictionary-attack check: an attacker who knows the ID format hashes every candidate with plain SHA-256.
# Against our HMAC output this finds nothing; against plain SHA-256 output it would match every row.
import hashlib

sample_ids = transformed["resale_identifier"].head(1000)
plain_sha = set(sample_ids.map(lambda v: hashlib.sha256(v.encode()).hexdigest()))
print("Plain-SHA-256 guesses that match our hashes:", len(plain_sha & set(hashed["resale_identifier_hash"])))

Plain-SHA-256 guesses that match our hashes: 0


In [36]:
paths = write_output(hashed, "hashed", CFG)
print({k: str(v.relative_to(Path.cwd())) for k, v in paths.items()})

{'parquet': 'data/hashed/hashed.parquet', 'csv': 'data/hashed/hashed.csv'}


## 8. Pipeline summary

All five required output groups, with row counts reconciled from raw to hashed.

In [37]:
in_scope_raw = int(std_summary["rows_in_scope"].sum())
summary = pd.DataFrame([
    {"output_group": "Raw (in scope)", "rows": in_scope_raw, "location": "data/raw/"},
    {"output_group": "Cleaned",        "rows": len(cleaned),     "location": "data/cleaned/"},
    {"output_group": "Quarantined",    "rows": len(quarantined), "location": "data/quarantined/"},
    {"output_group": "Transformed",    "rows": len(transformed), "location": "data/transformed/"},
    {"output_group": "Hashed",         "rows": len(hashed),      "location": "data/hashed/"},
])
assert len(cleaned) + len(quarantined) == in_scope_raw, "cleaned + quarantined != raw in scope"
assert len(cleaned) == len(transformed) == len(hashed), "row count changed after cleaning"
print("✅ Reconciled: raw in scope = cleaned + quarantined; cleaned = transformed = hashed")
summary

✅ Reconciled: raw in scope = cleaned + quarantined; cleaned = transformed = hashed


,output_group,rows,location
0,Raw (in scope),113053,data/raw/
1,Cleaned,104783,data/cleaned/
2,Quarantined,8270,data/quarantined/
3,Transformed,104783,data/transformed/
4,Hashed,104783,data/hashed/
